# Cost Model Comparison (grid:astar)

Compare all cost models using `grid:astar` across weights 1, 10, 100.
This notebook generates 20 elevation samples and visualizes resulting paths.

In [1]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger as loguru_logger

# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [2]:
from passage.generate import find_path, load_tile_grid
from passage.pathfinding_utils import COST_MODELS

In [ ]:
# Parameters (papermill)
RESOLUTION = 512
NUM_SAMPLES = 20

In [ ]:
# Configuration
CONFIG_PATH = ROOT / "config" / "config.yaml"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

WEIGHTS = [1.0, 10.0, 100.0]
COST_MODELS_LIST = sorted(list(COST_MODELS))
TILE_SIZE_PX = 3600

print(f"Resolution: {RESOLUTION}")
print(f"Weights: {WEIGHTS}")
print(f"Cost models: {COST_MODELS_LIST}")

Resolution: 256
Weights: [1.0, 10.0, 100.0]
Cost models: ['absolute_elevation', 'distance', 'elevation', 'energy', 'slope_uphill']


In [4]:
# Load calibration data
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")
print(f"Number of tiles: {len(tile_stats_df)}")

# outputs cleared

Global elevation range: [-430.0, 8767.0]
Number of tiles: 23993


In [ ]:
import math
import random


def generate_random_crop(
    resolution: int,
    tile_stats_df: pd.DataFrame,
    tiles_dirpath: Path,
    global_min: float,
    global_max: float,
    seed: int | None = None,
) -> np.ndarray | None:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    half_res = resolution // 2
    half_tiles = max(1, math.ceil(half_res / TILE_SIZE_PX))

    for _ in range(10):
        target_elev = random.uniform(global_min, global_max)
        candidates = tile_stats_df[(tile_stats_df["min"] <= target_elev) & (tile_stats_df["max"] >= target_elev)]
        if candidates.empty:
            continue
        selected = candidates.sample(n=1).iloc[0]
        center_lat = int(selected["lat"])
        center_lon = int(selected["lon"])
        grid = load_tile_grid(tiles_dirpath, center_lat, center_lon, half_tiles=half_tiles)
        grid_h, grid_w = grid.shape
        if grid_h < resolution or grid_w < resolution:
            continue
        y1 = random.randint(0, grid_h - resolution)
        x1 = random.randint(0, grid_w - resolution)
        crop = grid[y1 : y1 + resolution, x1 : x1 + resolution]
        return crop
    return None


def get_marker_positions(resolution: int) -> tuple[tuple[int, int], tuple[int, int]]:
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    return (pos1, pos1), (pos2, pos2)

In [6]:
def compute_paths_for_crop(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int], cost_models: list[str], weights: list[float]
) -> dict:
    results = {}
    for cost_model_name in cost_models:
        results[cost_model_name] = {}
        for w in weights:
            solver_config = {
                "name": "grid:astar",
                "diagonal_movement": "always",
                "weight": float(w),
                "cost_model": cost_model_name,
                "heuristic": "euclidean",
            }
            path = find_path(crop, start, end, solver_config)
            results[cost_model_name][w] = path
    return results


def make_tensor_vis(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int]
) -> tuple[np.ndarray, tuple[float, float]]:
    # Use raw elevation with provided range for consistent normalization in visualization
    elev_min = float(np.min(crop))
    elev_max = float(np.max(crop))
    marker_channel = np.zeros_like(crop, dtype=np.float32)
    marker_channel[start] = 1.0
    marker_channel[end] = 1.0
    obstacles_channel = np.zeros_like(crop, dtype=np.float32)
    tensor_vis = np.stack([crop.astype(np.float32), marker_channel, obstacles_channel], axis=-1)
    return tensor_vis, (elev_min, elev_max)

In [7]:
from tqdm import tqdm


samples = []
records = []

for i in tqdm(range(NUM_SAMPLES)):
    crop = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=1000 + i)
    if crop is None:
        print(f"Sample {i + 1}: failed to generate crop")
        continue
    start, end = get_marker_positions(RESOLUTION)
    paths = compute_paths_for_crop(crop, start, end, COST_MODELS_LIST, WEIGHTS)
    samples.append({"index": i + 1, "crop": crop, "start": start, "end": end, "paths": paths})

    for cost_model_name in COST_MODELS_LIST:
        for w in WEIGHTS:
            path = paths[cost_model_name][w]
            records.append(
                {
                    "sample": i + 1,
                    "cost_model": cost_model_name,
                    "weight": w,
                    "path_length": len(path),
                    "found": len(path) > 0,
                }
            )


print(f"\nGenerated {len(samples)}/{NUM_SAMPLES} samples")

# outputs cleared

100%|██████████| 20/20 [00:36<00:00,  1.85s/it]


Generated 20/20 samples


In [8]:
results_df = pd.DataFrame(records)
results_df.head()

# outputs cleared

,sample,cost_model,weight,path_length,found
0,1,absolute_elevation,1.0,154,True
1,1,absolute_elevation,10.0,154,True
2,1,absolute_elevation,100.0,154,True
3,1,distance,1.0,154,True
4,1,distance,10.0,154,True


In [9]:
summary_df = (
    results_df.groupby(["cost_model", "weight"])
    .agg(
        mean_path_length=("path_length", "mean"),
        min_path_length=("path_length", "min"),
        max_path_length=("path_length", "max"),
        found_rate=("found", "mean"),
    )
    .reset_index()
)
summary_df

# outputs cleared

,cost_model,weight,mean_path_length,min_path_length,max_path_length,found_rate
0,absolute_elevation,1.0,154.10,154,156,1.0
1,absolute_elevation,10.0,159.80,154,199,1.0
2,absolute_elevation,100.0,166.55,154,257,1.0
3,distance,1.0,154.00,154,154,1.0
4,distance,10.0,154.00,154,154,1.0
5,distance,100.0,154.00,154,154,1.0
6,elevation,1.0,157.00,154,193,1.0
7,elevation,10.0,214.30,154,348,1.0
8,elevation,100.0,263.75,157,414,1.0
9,energy,1.0,155.85,154,172,1.0


In [10]:
from matplotlib.lines import Line2D

# Per-sample grid of elevation + path visualizations (rows = cost models, cols = weights)
for sample in samples:
    crop = sample["crop"]
    start = sample["start"]
    end = sample["end"]
    paths = sample["paths"]

    elev_min, elev_max = float(np.min(crop)), float(np.max(crop))

    rows = len(COST_MODELS_LIST)
    cols = len(WEIGHTS)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    if cols == 1:
        axes = axes.reshape(-1, 1)

    for r, cm in enumerate(COST_MODELS_LIST):
        for c, w in enumerate(WEIGHTS):
            ax = axes[r, c]
            last_im = ax.imshow(crop, cmap="terrain", vmin=elev_min, vmax=elev_max, origin="upper")
            path = paths[cost_model_name][w]
            if len(path) > 0:
                pa = np.array(path)
                ax.plot(pa[:, 1], pa[:, 0], color="red", linewidth=2)
            ax.plot(start[1], start[0], "o", color="lime", markersize=6)
            ax.plot(end[1], end[0], "x", color="cyan", markersize=6)
            ax.set_title(f"{cm} | w={w} | len={len(path)}", fontsize=9)
            ax.axis("off")

    # Figure-level legend
    legend_handles = [
        Line2D([0], [0], color="red", lw=2, label="Path"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="lime", markersize=6, label="Start"),
        Line2D([0], [0], marker="x", color="cyan", markersize=6, label="End"),
    ]
    fig.legend(handles=legend_handles, loc="upper right")
    fig.suptitle(f"Sample {sample['index']}")
    plt.tight_layout()
    plt.show()